In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity ,manhattan_distances
from scipy.spatial.distance import jaccard
from sklearn.neighbors import NearestNeighbors
#---------------------------------------------------------
from sklearn.model_selection import train_test_split
#---------------------------------------------------------
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder , MinMaxScaler , MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer 

In [3]:
origin_df = pd.read_csv('movie_metadata.csv')
print(origin_df.shape)
origin_df.head(5)

(5043, 28)


,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [4]:
df = origin_df[['movie_title','genres','director_name','actor_1_name','actor_2_name','actor_3_name','language','imdb_score','content_rating','title_year','duration']]

In [5]:
df.rename(index =df['movie_title'] ,inplace=True)
df.drop(columns=['movie_title'],inplace=True)
df.dropna(inplace=True)
df.shape

(4656, 10)

In [6]:
df['genres'] = df['genres'].str.replace('|',' genre_')
for col in ['director_name','actor_1_name','actor_2_name','actor_3_name']:
    df[col]=df[col].str.replace(' ','_')
df['Combined_Features'] = 'genre_'+df['genres']+' '+'director_'+df['director_name']+ ' ' +'star1_'+df['actor_1_name']+' '+'star2_'+df['actor_2_name']+' '+'star3_'+df['actor_3_name']

# preprocessing

**CountVectorizer()**

In [8]:
vector = CountVectorizer()
vector_data = vector.fit_transform(df['Combined_Features'])

In [9]:
vector.get_feature_names_out()

array(['_abrams', '_adler', '_allen', ..., 'woon_kim', 'wurmfeld',
       'yin_lee'], shape=(10336,), dtype=object)

In [10]:
vector_data          # we need to change the type of this matrix to array for buliding new df

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 33608 stored elements and shape (4656, 10336)>

In [11]:
vect_df = pd.DataFrame(vector_data.toarray(),columns=vector.get_feature_names_out(),index=df.index)
vect_df.head(3)

,_abrams,_adler,_allen,_anderson,_atwell,_auman,_avildsen,_bailey,_barry,_bassett,...,ven_kelly,victor,vosloo,wai_wong,whitfield,won_ha,wook_park,woon_kim,wurmfeld,yin_lee
Avatar,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pirates of the Caribbean: At World's End,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Spectre,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### cancat duration, year and rate

In [12]:
df_add = df[['language','imdb_score','content_rating','title_year','duration']]
df_add.head(3)

,language,imdb_score,content_rating,title_year,duration
Avatar,English,7.9,PG-13,2009.0,178.0
Pirates of the Caribbean: At World's End,English,7.1,PG-13,2007.0,169.0
Spectre,English,6.8,PG-13,2015.0,148.0


In [13]:
rating_cat = df_add[['content_rating']]
language_cat = df_add[['language']]

# OrdinalEncoder()

In [14]:
# چون دسته بندی ها واقعا به هم نزدیکن
ordinal_encoder = OrdinalEncoder()
rating_encoded = ordinal_encoder.fit_transform(rating_cat)
rating_encoded

array([[7.],
       [7.],
       [7.],
       ...,
       [5.],
       [7.],
       [6.]], shape=(4656, 1))

In [16]:
ordinal_encoder.categories_

[array(['Approved', 'G', 'GP', 'M', 'NC-17', 'Not Rated', 'PG', 'PG-13',
        'Passed', 'R', 'TV-14', 'TV-G', 'TV-PG', 'Unrated', 'X'],
       dtype=object)]

In [17]:
cat_language = OneHotEncoder()
languages_encoded = cat_language.fit_transform(language_cat)

In [18]:
cat_language.categories_

[array(['Aboriginal', 'Arabic', 'Aramaic', 'Bosnian', 'Cantonese',
        'Chinese', 'Czech', 'Danish', 'Dari', 'Dutch', 'English',
        'Filipino', 'French', 'German', 'Greek', 'Hebrew', 'Hindi',
        'Hungarian', 'Indonesian', 'Italian', 'Japanese', 'Kazakh',
        'Korean', 'Mandarin', 'Maya', 'Mongolian', 'Norwegian', 'Persian',
        'Polish', 'Portuguese', 'Romanian', 'Russian', 'Spanish',
        'Swedish', 'Thai', 'Vietnamese', 'Zulu'], dtype=object)]

In [20]:
languages_encoded.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4656, 37))

In [21]:
cat_language_df = pd.DataFrame(rating_encoded,columns=ordinal_encoder.get_feature_names_out(),index=df.index)
cat_rate_df = pd.DataFrame(languages_encoded.toarray(),columns=cat_language.get_feature_names_out(),index=df.index)

In [34]:
print(cat_language_df.shape)
cat_language_df.head(3)

(4656, 1)


,content_rating
Avatar,7.0
Pirates of the Caribbean: At World's End,7.0
Spectre,7.0


In [35]:
print(cat_rate_df.shape)
cat_rate_df.head(3)

(4656, 37)


,language_Aboriginal,language_Arabic,language_Aramaic,language_Bosnian,language_Cantonese,language_Chinese,language_Czech,language_Danish,language_Dari,language_Dutch,...,language_Persian,language_Polish,language_Portuguese,language_Romanian,language_Russian,language_Spanish,language_Swedish,language_Thai,language_Vietnamese,language_Zulu
Avatar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Pirates of the Caribbean: At World's End,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Spectre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Scaling

**MinMaxScaler()**

In [36]:
df_scale = df_add[['imdb_score','title_year','duration']]
scale = MinMaxScaler()
scaler=scale.fit_transform(df_scale)

In [39]:
scale.get_feature_names_out()

array(['imdb_score', 'title_year', 'duration'], dtype=object)

In [37]:
scaler

array([[0.81818182, 0.92134831, 0.50967742],
       [0.71428571, 0.8988764 , 0.48064516],
       [0.67532468, 0.98876404, 0.41290323],
       ...,
       [0.62337662, 0.94382022, 0.24193548],
       [0.61038961, 0.95505618, 0.25806452],
       [0.64935065, 0.86516854, 0.22580645]], shape=(4656, 3))

In [42]:
scale_df = pd.DataFrame(scaler,columns=scale.get_feature_names_out(),index=df.index)
print(scale_df.shape)
scale_df.head(3)

(4656, 3)


,imdb_score,title_year,duration
Avatar,0.818182,0.921348,0.509677
Pirates of the Caribbean: At World's End,0.714286,0.898876,0.480645
Spectre,0.675325,0.988764,0.412903


vect_df
cat_language_df
cat_rate_df
scale_df

In [47]:
Data = pd.concat([vect_df,cat_language_df,cat_rate_df,scale_df],axis=1)  
print(Data.shape)
Data.head(5)

(4656, 10377)


,_abrams,_adler,_allen,_anderson,_atwell,_auman,_avildsen,_bailey,_barry,_bassett,...,language_Romanian,language_Russian,language_Spanish,language_Swedish,language_Thai,language_Vietnamese,language_Zulu,imdb_score,title_year,duration
Avatar,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.818182,0.921348,0.509677
Pirates of the Caribbean: At World's End,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.714286,0.898876,0.480645
Spectre,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.675325,0.988764,0.412903
The Dark Knight Rises,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.896104,0.955056,0.464516
John Carter,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.649351,0.955056,0.361290


## Metrices

**cosine_similarity**

In [52]:
cosine_sim_matrix = cosine_similarity(Data)
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, index=Data.index, columns=Data.index)

In [53]:
cosine_sim_df.head()

,Avatar,Pirates of the Caribbean: At World's End,Spectre,The Dark Knight Rises,John Carter,Spider-Man 3,Tangled,Avengers: Age of Ultron,Harry Potter and the Half-Blood Prince,Batman v Superman: Dawn of Justice,...,Pink Flamingos,Clean,The Circle,Primer,Cavite,El Mariachi,The Mongol King,Newlyweds,Shanghai Calling,My Date with Drew
Avatar,1.000000,0.916255,0.899344,0.883078,0.924777,0.891399,0.841327,0.925166,0.877214,0.917523,...,0.776921,0.870727,0.823603,0.890759,0.845828,0.873533,0.872684,0.833149,0.864968,0.867135
Pirates of the Caribbean: At World's End,0.916255,1.000000,0.914521,0.897549,0.906634,0.906596,0.855131,0.906874,0.891597,0.899457,...,0.789458,0.885779,0.836728,0.871949,0.859733,0.888729,0.887255,0.846754,0.879698,0.881628
Spectre,0.899344,0.914521,1.000000,0.914476,0.906820,0.906642,0.837006,0.906999,0.872504,0.899495,...,0.789178,0.885794,0.836992,0.889151,0.883322,0.902274,0.887414,0.847355,0.880048,0.881953
The Dark Knight Rises,0.883078,0.897549,0.914476,1.000000,0.889603,0.889318,0.819291,0.890254,0.854061,0.882715,...,0.790308,0.885224,0.838179,0.888970,0.883516,0.901637,0.887681,0.847672,0.879581,0.881998
John Carter,0.924777,0.906634,0.906820,0.889603,1.000000,0.898909,0.829733,0.932746,0.864782,0.924932,...,0.782226,0.878518,0.829566,0.898583,0.852783,0.881497,0.879927,0.839846,0.872618,0.874413


In [ ]:
plt.figure(figsize=(200,200))
sns.heatmap(cosine_sim_matrix)
plt.savefig('figure.png',dpi=300)

In [64]:
print(cosine_sim_df.columns.tolist())

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz the Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lante

In [70]:
cosine_sim_df.columns = cosine_sim_df.columns.str.replace('\xa0', '', regex=False).str.strip()
cosine_sim_df.index = cosine_sim_df.index.str.replace('\xa0', '', regex=False).str.strip()

def get_top_5_similar_movies(movie_title, cosine_sim_df):
    movie_title = movie_title.strip()
    if movie_title not in cosine_sim_df.index:
        return f"Movie '{movie_title}' not found. Please check the spelling."
    
    sim_scores = cosine_sim_df[movie_title]
    
    # مرتب‌سازی و گرفتن ۵ تای اول (بدون خود فیلم)
    similar_movies = sim_scores.sort_values(ascending=False).iloc[1:6].index.tolist()
    return similar_movies


    
movie_name=input("Please Enter your movie_name : ")
print(get_top_5_similar_movies(movie_name, cosine_sim_df))

Please Enter your movie_name :  The Dark Knight Rises


['Inception', 'RocknRolla', 'Sin City: A Dame to Kill For', 'Memento', 'No Escape']
